In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from glob import  glob
import os
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
!unzip "/content/drive/MyDrive/lise makale/200X_CNN_data.zip"

In [ ]:
import numpy as np
import pandas as pd
import os
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
import matplotlib.pyplot as plt
from sklearn import model_selection
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

import cv2
import random


In [ ]:
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import RMSprop
from keras import regularizers
from keras.layers.core import Dropout
from keras.applications.vgg16 import VGG16
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.losses import BinaryCrossentropy

from sklearn.metrics import precision_score, f1_score, recall_score, accuracy_score, roc_auc_score

In [ ]:
train = '/content/200X_Data_CNN_DATA/train'
test =  '/content/200X_Data_CNN_DATA/test'

classes = ["adenosis","fibroadenoma","papillary_carcinoma","tubular_adenoma","ductal_carcinoma","lobular_carcinoma","mucinous_carcinoma","papillary_carcinoma"]

In [ ]:
train_data = ImageDataGenerator(rescale=1./255,
                                   zoom_range=0.3,validation_split=0.15,
                                   horizontal_flip=True)
test_data = ImageDataGenerator(rescale=1./255)



traindata = train_data.flow_from_directory(train,
                                                batch_size=8,subset = "training",
                                                target_size=(224,224),
                                                shuffle=True,
                                                color_mode='rgb',
                                                class_mode='categorical')

validationdata = train_data.flow_from_directory(train,
                                                batch_size=8,
                                                target_size=(224,224),subset = "validation",
                                                shuffle=True,
                                                color_mode='rgb',
                                                class_mode='categorical')

testdata = test_data.flow_from_directory(test,
                                                batch_size=8,
                                                target_size=(224,224),
                                                shuffle=False,
                                                color_mode='rgb',
                                                class_mode='categorical')


Found 784 images belonging to 8 classes.
Found 135 images belonging to 8 classes.
Found 398 images belonging to 8 classes.


**VGG16**

In [ ]:
#vgg = VGG16(include_top=False, weights='imagenet', input_shape=(224,224,3))
#
#x = Flatten()(vgg.output)
#x = Dense(256, activation='relu')(x)
#output = Dense(8, activation='softmax')(x)


vgg = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# VGG16 tabanına ekstra katmanlar ekleyin
x = vgg.output
x = Flatten()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)  # Dropout düzenlemesi ekleyin
x = Dense(256, activation='relu')(x)
output = Dense(8, activation='softmax')(x)

for layer in vgg.layers:
    layer.trainable = False


vgg16Model = Model(inputs=vgg.input, outputs=output)

vgg16Model.summary()

In [ ]:
steps_per_epoch = traindata.n // traindata.batch_size
validation_steps = validationdata.n // validationdata.batch_size


In [ ]:
vgg16Model.compile(optimizer=SGD(learning_rate=0.0001),loss = "categorical_crossentropy",metrics = ["accuracy"])

hist = vgg16Model.fit(traindata,steps_per_epoch=validation_steps,
                 validation_data=validationdata,validation_steps=validation_steps,
                 epochs=300)


In [ ]:
score_val_model = vgg16Model.evaluate(validationdata, steps = len(validationdata), verbose = 0)
print('val loss:', score_val_model[0])
print('val accuracy:', score_val_model[1])


val loss: 2.0819149017333984
val accuracy: 0.42222222685813904


In [ ]:
score_test_model = vgg16Model.evaluate(testdata, steps = len(testdata), verbose = 0)
print('Test loss:', score_test_model[0])
print('Test accuracy:', score_test_model[1])


Test loss: 1.4257175922393799
Test accuracy: 0.5226130485534668


In [ ]:
import keras
# Calling `save('my_model')` creates a SavedModel folder `my_model`.
vgg16Model.save("/content/drive/MyDrive/lise makale/VGG16/train 1/vgg16.h5")


In [ ]:
VGGmodel16 = keras.models.load_model('/content/drive/MyDrive/lise makale/VGG16/train 1/vgg16.h5')

In [ ]:
preds_vgg16 = VGGmodel16.predict(testdata,steps=len(testdata))

50/50 [==============================] - 12s 215ms/step


In [ ]:
import json,codecs
with open("history.json","w") as f:
    json.dump(hist.history,f)
with codecs.open("history.json","r",encoding="utf-8") as f:
    n = json.loads(f.read())

In [ ]:
plt.plot(n["loss"],label = "training loss")
plt.plot(n["val_loss"],label = "validation loss")
plt.savefig('LOSS.png')
plt.legend()
plt.show()


plt.plot(n["accuracy"],label = "training acc")
plt.plot(n["val_accuracy"],label = "validation acc")
plt.savefig('accuracy.png')
plt.legend()
plt.show()



In [ ]:
shutil.copy("/content/LOSS.png","/content/drive/MyDrive/lise makale/VGG16/train 1")
shutil.copy("/content/accuracy.png","/content/drive/MyDrive/lise makale/VGG16/train 1")

'/content/drive/MyDrive/lise makale/VGG16/train 1/accuracy.png'

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns


Y_pred = VGGmodel16.predict(testdata)
y_pred = np.argmax(Y_pred, axis=1)

confusion_mtx = confusion_matrix(testdata.classes, y_pred)
f,ax = plt.subplots(figsize=(10, 10))

sns.heatmap(confusion_mtx, annot=True, linewidths=0.01,cmap="Blues",linecolor="gray", fmt= '.1f',ax=ax)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.savefig('Confusion Matrix.png')
plt.show()

In [ ]:
print("Micro Precision: %f "%precision_score(testdata.classes, y_pred, average='micro'))
print("Micro Recall: %f "%recall_score(testdata.classes, y_pred, average='micro'))
print("Micro F1: %f"% f1_score(testdata.classes, y_pred, average='micro'))
#print("AUC Score: %f"%roc_auc_score(y_true, y_pred, average='micro'))
print("\n")

print("Acuracy: %f "%accuracy_score(testdata.classes, y_pred))
print("Macro Precision: %f "%precision_score(testdata.classes, y_pred, average='macro'))
print("Macro Recall: %f "%recall_score(testdata.classes, y_pred, average='macro'))
print("Macro F1: %f"% f1_score(testdata.classes, y_pred, average='macro'))
#print("AUC Score: %f"%roc_auc_score(testdata.classes, y_pred, average='macro'))
print("\n")

print("Weighted Precision: %f "%precision_score(testdata.classes, y_pred, average='weighted'))
print("Weighted Recall: %f "%recall_score(testdata.classes, y_pred, average='weighted'))
print("Weighted F1: %f"% f1_score(testdata.classes, y_pred, average='weighted'))
#print("ROC_AUC Score: %f"%roc_auc_score(testdata.classes, y_pred, average='weighted'))

Micro Precision: 0.522613 
Micro Recall: 0.522613 
Micro F1: 0.522613


Acuracy: 0.522613 
Macro Precision: 0.544892 
Macro Recall: 0.510525 
Macro F1: 0.506924


Weighted Precision: 0.554116 
Weighted Recall: 0.522613 
Weighted F1: 0.517091
